# Lesson 07 - Planning Design Pattern

Dis notebook dey show how to use **Planning Design Pattern** for AI agents with Microsoft Agent Framework.
You go learn how to break complex travel requests into structured subtasks, assign dem to specialist agents,
and run the plan wey come out — all na with structured output powered by Pydantic models.


## Setup


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Task Decomposition

Task decomposition na di core of di planning design pattern. Instead of to ask one agent to
handle one big complex request from start to finish, we go break di problem into smaller, well-defined **subtasks**.
Each subtask dem assign to one specialist agent (like flights, hotels, activities) wey get clear
priorities and dependency ordering.

Dis approach get plenty benefits:
- **Clarity**: each subtask get only one responsibility.
- **Parallelism**: independent subtasks fit run at di same time.
- **Reliability**: failures dey isolated to individual subtasks.
- **Budget tracking**: costs dey estimated per subtask and dem go roll up.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Creating a Planning Agent with Structured Output

Di planning agent dey work as **front desk coordinator**. When you give am one high-level travel request, e go produce one structured `TravelPlan` — wey go break down di request into smaller tasks, arrange priority dem, and sabi wetin depend on wetin make sure say concierge or execution layer fit run di work.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Executing a Plan with Specialist Tools

Once di front desk agent don produce structured plan, di **concierge agent** go execute am.  
Each specialist tool dey handle one kain subtask category (flights, hotels, activities). Di concierge  
go waka through di plan subtasks for dependency order come send each one go di correct tool.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Summary

For dis lesson, you learn di **Planning Design Pattern** for AI agents:

1. **Task Decomposition** — One front desk planning agent go break one complex travel request into structured subtasks with Pydantic models, den e go assign each subtask to one specialist agent with priorities and dependencies.
2. **Structured Output** — By passing one `response_format` di agent go return one validated `TravelPlan` object instead of free-form text, wey go make downstream processing reliable.
3. **Plan Execution** — One concierge agent go waka through di subtasks using specialist tools (`book_flight`, `reserve_hotel`, `book_activity`) to carry out di plan and report results.

Dis pattern dey separate *wetin to do* (planning) from *how to do am* (execution), e dey make agents more modular, testable, and easy to extend.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:  
Dis document don translate wit AI translation service wey dem dey call [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg sabi say sometimes machine fit make mistake or no translate well. Di original document wey dem write for dia own language na di correct one wey you suppose trust. If na serious tori, e better make professional person translate am. We no dey responsible for any yawa wey fit happen if people misunderstand or no take am well because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
